# Phase 2K - Fold composition and a feasibility pilot

**Run on:** Kaggle, CPU only. A few minutes. No training.

1. The class composition of every fold construction the paper trains on.
2. Whether a density-matched deletion control is feasible for Phase 2N v2.

In [ ]:
import subprocess, sys, os, json, shutil
import numpy as np, pandas as pd, cv2, sklearn
from tqdm.auto import tqdm
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold, StratifiedKFold

SEED, N_FOLDS = 42, 5
IN_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if IN_KAGGLE else "/content"
RESULTS_DIR = f"{WORK}/fyp_phase2k_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
RESULTS = {"seed": SEED, "n_folds": N_FOLDS, "sklearn": sklearn.__version__,
           "numpy": np.__version__}
print("sklearn", sklearn.__version__, "| numpy", np.__version__)

def save_json():
    with open(f"{RESULTS_DIR}/results.json", "w") as f:
        json.dump(RESULTS, f, indent=2, default=float)

FOLDERS = {"Benign": "Bengin cases", "Malignant": "Malignant cases",
           "Normal": "Normal cases"}
DATA_ROOT = None
if os.path.isdir("/kaggle/input"):
    hits = [d for d, _, _ in os.walk("/kaggle/input")
            if os.path.basename(d) == "Malignant cases"]
    if hits:
        DATA_ROOT = os.path.dirname(hits[0])
if DATA_ROOT is None:
    import kagglehub
    DL = kagglehub.dataset_download("hamdallak/the-iqothnccd-lung-cancer-dataset")
    DATA_ROOT = os.path.dirname([d for d, _, _ in os.walk(DL)
                                 if os.path.basename(d) == "Malignant cases"][0])

SPLIT_URL = ("https://raw.githubusercontent.com/haseebkhan9081/"
             "iqothnccd-leakage-audit/main/split_seed42.csv")
subprocess.run(["wget", "-q", "-O", f"{WORK}/split_seed42.csv", SPLIT_URL], check=True)
df = pd.read_csv(f"{WORK}/split_seed42.csv")
df["path"] = [os.path.join(DATA_ROOT, FOLDERS[l], f)
              for l, f in zip(df["label"], df["file"])]
assert all(os.path.exists(p) for p in df["path"])
y_all = df["y"].to_numpy()
groups = df["group"].to_numpy()
POP = np.bincount(y_all, minlength=3) / len(y_all)
print("images", len(df), "| groups", len(set(groups)), "| population", POP.round(4))

## 1. Class composition of each fold construction

Per fold: test-set prevalence (benign, malignant, normal), the largest absolute
deviation from the population in percentage points, and the total variation
distance from the population (half the L1 distance, in points). Training-set
prevalence is reported as well, since a test fold rich in one class leaves the
training set poor in it.

In [ ]:
CONSTRUCTIONS = {
    "image-level": list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
                        .split(df, y_all)),
    "grouped-A (GroupKFold)": list(GroupKFold(N_FOLDS).split(df, y_all, groups)),
    "grouped-B (StratifiedGroupKFold)": list(
        StratifiedGroupKFold(N_FOLDS, shuffle=True, random_state=SEED)
        .split(df, y_all, groups)),
}

comp = {}
for name, folds in CONSTRUCTIONS.items():
    rows = []
    for k, (tr, te) in enumerate(folds):
        pte = np.bincount(y_all[te], minlength=3) / len(te)
        ptr = np.bincount(y_all[tr], minlength=3) / len(tr)
        rows.append({"fold": k, "n_test": int(len(te)),
                     "test_prev": pte.round(4).tolist(),
                     "train_prev": ptr.round(4).tolist(),
                     "max_abs_dev_pp": round(float(100 * np.abs(pte - POP).max()), 2),
                     "tvd_pp": round(float(50 * np.abs(pte - POP).sum()), 2)})
    comp[name] = rows
    print(name)
    for r in rows:
        print(f"  fold {r['fold']}: test {r['test_prev']}  n={r['n_test']}  "
              f"max dev {r['max_abs_dev_pp']:5.1f} pp  TVD {r['tvd_pp']:5.1f} pp")
    devs = [r["max_abs_dev_pp"] for r in rows]
    print(f"  worst fold {max(devs):.1f} pp | mean {np.mean(devs):.1f} pp | "
          f"benign share range {min(r['test_prev'][0] for r in rows):.3f}"
          f"-{max(r['test_prev'][0] for r in rows):.3f}\n")
RESULTS["population_prev"] = POP.round(4).tolist()
RESULTS["fold_composition"] = comp

# Guard: construction B must equal the prevalence archived by the trained run.
ARCHIVED_B = [[0.1163, 0.3721, 0.5116], [0.0388, 0.4729, 0.4884],
              [0.3115, 0.4044, 0.2842], [0.0757, 0.6733, 0.251],
              [0.0601, 0.5665, 0.3734]]
got_B = [r["test_prev"] for r in comp["grouped-B (StratifiedGroupKFold)"]]
RESULTS["construction_B_matches_archive"] = bool(np.allclose(got_B, ARCHIVED_B, atol=1e-4))
print("construction B reproduces the archived trained folds:",
      RESULTS["construction_B_matches_archive"])
save_json()

## 2. Feasibility of a density-matched deletion control

Density of a training image = how many OTHER training images sit at cosine >=
threshold to it. It uses the training partition only, so it is defined without
looking at the test fold. Deduplication removes the "contaminated" images (those
with a test neighbour). The question: does the CLEAN pool (no test neighbour)
contain enough images in each class x density bin to draw a same-size,
same-class, same-density sample?

In [ ]:
thumbs = []
for p in tqdm(df["path"], desc="thumbnails"):
    g = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2GRAY)
    g = cv2.resize(g, (64, 64), interpolation=cv2.INTER_AREA).astype(np.float32).ravel()
    g -= g.mean(); n = np.linalg.norm(g)
    thumbs.append(g / n if n > 0 else g)
T = np.stack(thumbs); S = T @ T.T; np.fill_diagonal(S, 0.0)

def density_bins(d):
    # quantile-free fixed bins so folds are comparable
    return np.digitize(d, [1, 3, 6, 11, 21])   # 0, 1-2, 3-5, 6-10, 11-20, 21+

pilot = []
for thr in (0.98, 0.94):
    for k, (tr, te) in enumerate(CONSTRUCTIONS["image-level"]):
        contaminated = (S[np.ix_(tr, te)] >= thr).any(axis=1)
        dens = (S[np.ix_(tr, tr)] >= thr).sum(axis=1)
        b = density_bins(dens)
        cls = y_all[tr]
        need, have, short = 0, 0, 0
        for c in range(3):
            for bb in range(6):
                n_need = int(((cls == c) & (b == bb) & contaminated).sum())
                n_have = int(((cls == c) & (b == bb) & ~contaminated).sum())
                need += n_need; short += max(0, n_need - n_have)
        frac = 1 - short / need if need else 1.0
        row = {"threshold": thr, "fold": k, "n_removed": int(contaminated.sum()),
               "clean_pool": int((~contaminated).sum()),
               "density_removed_median": float(np.median(dens[contaminated])),
               "density_clean_median": float(np.median(dens[~contaminated])),
               "shortfall": short, "matchable_fraction": round(frac, 3)}
        pilot.append(row)
        print(f"thr {thr} fold {k}: remove {row['n_removed']:3d} from clean pool "
              f"{row['clean_pool']:3d} | density median removed "
              f"{row['density_removed_median']:.0f} vs clean "
              f"{row['density_clean_median']:.0f} | exact match possible for "
              f"{100*frac:.0f}% (shortfall {short})")
RESULTS["density_match_pilot"] = pilot
save_json()
shutil.make_archive(f"{WORK}/fyp_phase2k_results", "zip", RESULTS_DIR)
print("done")